# IDX Exchange DA Internship - Week Six - Feature Engineering and Market Metrics

Submit a .py script demonstrating all engineered metrics (price ratio, close-to-original-list ratio, PPSF, days
on market, YrMo, listing-to-contract days, contract-to-close days), with a sample output table showing the
new columns populated correctly. Include at least one segmented summary table grouped by
PropertyType or CountyOrParish.

------

In [95]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

This week focuses on feature engineering new columns in order to improve our analysis. Most of these engineered columns will be key market metrics, such as price change ratios, price per square foot, and days between status changes. Additionally, the MLS data will be joined with a California school district dataset to get more accuracte school district information. Lastly, segment analysis will be performed on several features in order to observe the trends of the newly created metrics.

## Import Data

In [96]:
# Import the cleaned datasets from weeks 4-5

listings = pd.read_csv("listings_cleaned.csv", low_memory=False)
sold = pd.read_csv("sold_cleaned.csv", low_memory=False)

In [97]:
# Some ClosePrice and CloseDate values in listings are "Unknown" or "N/A" because not all properties have been closed
# Turn those into nulls so that we can still do arithmetic with the columns

listings["ClosePrice"] = listings["ClosePrice"].replace({"Unknown": np.nan}).astype('Float64')
listings["CloseDate"] = pd.to_datetime(listings["CloseDate"].replace({"Unknown": np.nan}))
listings["PurchaseContractDate"] = pd.to_datetime(listings["PurchaseContractDate"].replace({"Unknown": np.nan}))
listings["ListingContractDate"] = pd.to_datetime(listings["ListingContractDate"].replace({"Unknown": np.nan}))
sold["PurchaseContractDate"] = pd.to_datetime(sold["PurchaseContractDate"].replace({"Unknown": np.nan}))
sold["ListingContractDate"] = pd.to_datetime(sold["ListingContractDate"].replace({"Unknown": np.nan}))
sold["CloseDate"] = pd.to_datetime(sold["CloseDate"])

## Adding Market Metric Features

### Price Ratios

In [98]:
# This new feature captures the full price reduction history 

listings["OriginPriceRatio"] = listings["ClosePrice"] / listings["OriginalListPrice"]
sold["OriginPriceRatio"] = sold["ClosePrice"] / sold["OriginalListPrice"]

In [99]:
# This new feature measures negotiation strength

listings["PriceRatio"] = listings["ClosePrice"] / listings["ListPrice"]
sold["PriceRatio"] = sold["ClosePrice"] / sold["ListPrice"]

### Price Per Square Foot

In [100]:
# This new feature normalizes price across sizes

listings["PricePerSquareFoot"] = listings["ClosePrice"] / listings["LivingArea"]
sold["PricePerSquareFoot"] = sold["ClosePrice"] / sold["LivingArea"]

### Dates

Drilling down on the CloseDate column allows us to do time series analysis.

In [101]:
# Year column

listings["Year"] = listings["CloseDate"].dt.year
sold["Year"] = sold["CloseDate"].dt.year

In [102]:
# Month column

listings["Month"] = listings["CloseDate"].dt.month
sold["Month"] = sold["CloseDate"].dt.month

In [103]:
# Year-Month column

listings["YearMonth"] = listings["CloseDate"].dt.to_period('M')
sold["YearMonth"] = sold["CloseDate"].dt.to_period('M')

### Contract Days

In [104]:
# This new feature measures time from listing to accepted offer

listings["ListToContract"] = (listings["PurchaseContractDate"] - listings["ListingContractDate"]).dt.days
sold["ListToContract"] = (sold["PurchaseContractDate"] - sold["ListingContractDate"]).dt.days

In [105]:
# This new features measures escrow and closing period duration

listings["ContractToClose"] = (listings["CloseDate"] - listings["PurchaseContractDate"]).dt.days
sold["ContractToClose"] = (sold["CloseDate"] - sold["PurchaseContractDate"]).dt.days

### New Feature Validation

In this section, we check whether the new metrics have populated correctly. Note that the listings column has many missing values. This is because the majority of properties in the listings dataset have not yet been closed, and thus don't have the "ClosePrice" or "CloseDate" information necessary to compute these metrics. Thus, these null rows represent unsold properties. Appropriately, the sold dataset is completely populated since it has all of these neccesary features. 


Another interesting observation is that many rows in the sold dataset have an OriginPriceRatio and PriceRatio of 1.0, meaning the the price the property was listed at was the same it was sold for. Additionally, rows with no change between OriginPriceRatio and PriceRatio suggest that these properties were either sold the first time they were listed, or were listed multiple times with no change in listing price.

In [106]:
# New metrics, listings dataset

listings.loc[60:70, ["OriginPriceRatio", "PriceRatio", "PricePerSquareFoot", "YearMonth", "ListToContract", "ContractToClose"]]

,OriginPriceRatio,PriceRatio,PricePerSquareFoot,YearMonth,ListToContract,ContractToClose
60,<NA>,<NA>,<NA>,NaT,NaN,NaN
61,<NA>,<NA>,<NA>,NaT,NaN,NaN
62,0.98245,0.98245,674.06518,2024-04,72.0,26.0
63,<NA>,<NA>,<NA>,NaT,NaN,NaN
64,<NA>,<NA>,<NA>,NaT,84.0,NaN
65,<NA>,<NA>,<NA>,NaT,NaN,NaN
66,<NA>,<NA>,<NA>,NaT,83.0,NaN
67,0.953846,0.953846,291.353383,2024-03,70.0,2.0
68,1.016419,1.016419,751.879699,2024-04,75.0,17.0
69,<NA>,<NA>,<NA>,NaT,NaN,NaN


In [107]:
# New metrics, sold dataset

sold.loc[60:70, ["OriginPriceRatio", "PriceRatio", "PricePerSquareFoot", "YearMonth", "ListToContract", "ContractToClose"]]

,OriginPriceRatio,PriceRatio,PricePerSquareFoot,YearMonth,ListToContract,ContractToClose
60,1.003377,1.003377,517.980017,2024-01,0.0,0.0
61,1.000000,1.000000,390.434358,2024-01,0.0,0.0
62,1.007430,1.007430,550.078783,2024-01,0.0,0.0
63,1.000000,1.000000,554.177006,2024-01,0.0,0.0
64,1.000000,1.000000,1304.784209,2024-01,0.0,22.0
65,1.000000,1.000000,599.736181,2024-01,51.0,9.0
66,0.875000,0.875000,187.783739,2024-01,10.0,69.0
67,1.008688,1.008688,571.114072,2024-01,0.0,0.0
68,1.011304,1.011304,532.265351,2024-01,0.0,0.0
69,1.000000,1.000000,545.454545,2024-01,0.0,0.0


## Improved School Districts

In our current dataset, the HighSchoolDistrict feature has a high amount of "Unknown" values. We can improve this feature by joining our dataset with the California School District Areas 2024-25 dataset (https://data.ca.gov/@california-department-of-education/california-school-district-areas-2024-25). This dataset has the name of California school districts, the type of district (high, elementary, unified), enrollment counts, demographic data, and the longitude/latitude that the district encompasses. We can take the longitude and latitude of each property in our MLS dataset and match this to the school districts in this new dataset to further enrich our features.

In [108]:
schools = gpd.read_file("../DS_Internship/California_School_District_Areas_2024-25.geojson")

schools.columns

Index(['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName',
       'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh',
       'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'CongressUS',
       'SenateCA', 'AssemblyCA', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter',
       'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount',
       'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct',
       'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount',
       'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount',
       'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi',
       'Shape__Area', 'Shape__Length', 'geometry'],
      dtype='object')

In [109]:
# A lot of these columns are just metadata. Keep only the relevant columns

cols_to_keep = ["DistrictName", "DistrictType", "geometry"]
schools = schools[cols_to_keep]

In [110]:
# Merging the dataset naively creates duplicate rows where one property is matched to both its elementary
# and high school district, creating two rows for one property. To fix this, we must sequentially join the
# data together based on district type (high, elementary, unified).

### High School Distict Joining
geo_high = schools[schools["DistrictType"] == "High"]

# Listings
geometry = gpd.points_from_xy(listings["Longitude"], listings["Latitude"])
gdf_points = gpd.GeoDataFrame(listings, geometry=geometry, crs="EPSG:4326")
geo_high = geo_high.to_crs(gdf_points.crs)
geo_high_list = gpd.sjoin(gdf_points, geo_high, how="left", predicate="within")

# Sold
geometry = gpd.points_from_xy(sold["Longitude"], sold["Latitude"])
gdf_points = gpd.GeoDataFrame(sold, geometry=geometry, crs="EPSG:4326")
geo_high = geo_high.to_crs(gdf_points.crs)
geo_high_sold = gpd.sjoin(gdf_points, geo_high, how="left", predicate="within")



### Elementary School District Joining
geo_elem = schools[schools["DistrictType"] == "Elementary"]

# Listings
geometry = gpd.points_from_xy(listings["Longitude"], listings["Latitude"])
gdf_points = gpd.GeoDataFrame(listings, geometry=geometry, crs="EPSG:4326")
geo_elem = geo_elem.to_crs(gdf_points.crs)
geo_elem_list = gpd.sjoin(gdf_points, geo_elem, how="left", predicate="within")

# Sold
geometry = gpd.points_from_xy(sold["Longitude"], sold["Latitude"])
gdf_points = gpd.GeoDataFrame(sold, geometry=geometry, crs="EPSG:4326")
geo_elem = geo_elem.to_crs(gdf_points.crs)
geo_elem_sold = gpd.sjoin(gdf_points, geo_elem, how="left", predicate="within")



### Unified School District Joining
geo_unif = schools[schools["DistrictType"] == "Unified"]

# Listings
geometry = gpd.points_from_xy(listings["Longitude"], listings["Latitude"])
gdf_points = gpd.GeoDataFrame(listings, geometry=geometry, crs="EPSG:4326")
geo_unif = geo_unif.to_crs(gdf_points.crs)
geo_unif_list = gpd.sjoin(gdf_points, geo_unif, how="left", predicate="within")

# Sold
geometry = gpd.points_from_xy(sold["Longitude"], sold["Latitude"])
gdf_points = gpd.GeoDataFrame(sold, geometry=geometry, crs="EPSG:4326")
geo_unif = geo_unif.to_crs(gdf_points.crs)
geo_unif_sold = gpd.sjoin(gdf_points, geo_unif, how="left", predicate="within")

In [111]:
# Join the datasets sequentially

# Listings
geo_elem_list = geo_elem_list[cols_to_keep]
geo_unif_list = geo_unif_list[cols_to_keep]
geo_unif_list = geo_unif_list.add_suffix('_unif')
geo_list = geo_high_list.join(geo_elem_list, how="inner", lsuffix='_high', rsuffix='_elem').join(geo_unif_list, how="inner")

# Sold
geo_elem_sold = geo_elem_sold[cols_to_keep]
geo_unif_sold = geo_unif_sold[cols_to_keep]
geo_unif_sold = geo_unif_sold.add_suffix('_unif')
geo_sold = geo_high_sold.join(geo_elem_sold, how="inner", lsuffix='_high', rsuffix='_elem').join(geo_unif_sold, how="inner")

In [112]:
# Check that the join was successful

geo_list.loc[20:30, ["HighSchoolDistrict", "DistrictName_high", "DistrictName_elem", "DistrictName_unif"]]

,HighSchoolDistrict,DistrictName_high,DistrictName_elem,DistrictName_unif
20,San Diego Unified,NaN,NaN,San Diego Unified
21,Unknown,Sweetwater Union High,Chula Vista Elementary,NaN
22,Temecula Unified,NaN,NaN,Temecula Valley Unified
23,Sweetwater Union,Sweetwater Union High,Chula Vista Elementary,NaN
24,Huntington Beach Union High,Huntington Beach Union High,Huntington Beach City Elementary,NaN
25,Unknown,NaN,NaN,Hacienda la Puente Unified
26,Fullerton Joint Union High,Fullerton Joint Union High,Fullerton Elementary,NaN
27,Unknown,Acalanes Union High,Orinda Union,NaN
28,Unknown,NaN,NaN,Coachella Valley Unified
29,Newport Mesa Unified,NaN,NaN,Newport-Mesa Unified


In [113]:
# Check if there are any rows in the MLS data that could not be matched to any school district
# It seems that most of these rows show up as "Unknown" or "Other" 

condition_list = (geo_list["DistrictName_high"].isna() == True) & (geo_list["DistrictName_elem"].isna() == True) & (geo_list["DistrictName_unif"].isna() == True)
condition_sold = (geo_sold["DistrictName_high"].isna() == True) & (geo_sold["DistrictName_elem"].isna() == True) & (geo_sold["DistrictName_unif"].isna() == True)

geo_list[condition_list]["HighSchoolDistrict"].value_counts().sum() # 58 "Unknown", 30 "Other", 161 total
geo_sold[condition_sold]["HighSchoolDistrict"].value_counts().sum() # 72 "Unknown", "27 "Other", 149 total

np.int64(149)

In [114]:
# Fill Unknown HighSchoolDistrict with DistrictName

# If HighSchoolDistrict is "Unknown", O"ther", or null
condition_L = (geo_list["HighSchoolDistrict"] == "Unknown") | (geo_list["HighSchoolDistrict"] == "Other") | (geo_list["HighSchoolDistrict"] == np.nan)
condition_S = (geo_sold["HighSchoolDistrict"] == "Unknown") | (geo_sold["HighSchoolDistrict"] == "Other") | (geo_sold["HighSchoolDistrict"] == np.nan)

# Fill with new highschool district information. Fill remaining values with unified school district information
geo_list["HighSchoolDistrict"] = np.where(condition_L, geo_list["DistrictName_high"], geo_list["HighSchoolDistrict"])
geo_list["HighSchoolDistrict"] = np.where(condition_L, geo_list["DistrictName_unif"], geo_list["HighSchoolDistrict"])

geo_sold["HighSchoolDistrict"] = np.where(condition_S, geo_sold["DistrictName_high"], geo_sold["HighSchoolDistrict"])
geo_sold["HighSchoolDistrict"] = np.where(condition_S, geo_sold["DistrictName_unif"], geo_sold["HighSchoolDistrict"])

In [115]:
# Drop geometry, since it was only necessary for the join
# Drop DistrictType, since that information is now included in the title of the DistrictName columns

listings = geo_list.drop(columns = ["geometry_high", "geometry_elem", "geometry_unif",
                                     "DistrictType_high", "DistrictType_elem", "DistrictType_unif"])
sold = geo_sold.drop(columns = ["geometry_high", "geometry_elem", "geometry_unif",
                                     "DistrictType_high", "DistrictType_elem", "DistrictType_unif"])

## Segment Analysis

To better understand how our new metrics interact with the data, let's perform segment analysis. The key features to analyze this way are: PropertySubType, CountyOrParish, MLSAreaMajor, ListOfficeName, and BuyerOfficeName. Note once again that segment analyses generated with the listings dataset will likely have null values, since these metrics can only be generated for closed properties. Segment analyses generated with the sold dataset will not have this problem.

In [133]:
cols = ["OriginalListPrice", "ClosePrice", "OriginPriceRatio", "PriceRatio", "PricePerSquareFoot", "ListToContract", "ContractToClose"]

### PropertySubType

Analyzing PropertySubType gives us important information on how different kinds of residential properties sell. We can observe that the "Farm" and "MobileHome" rows are null in the listings segment, indicating that no farm or mobile home properties have been sold yet.

In [131]:
listings.groupby(["PropertySubType"])[cols].median()

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
PropertySubType,,,,,,,
BoatSlip,230000.0,191000.0,0.968031,0.971938,1375.0,10.0,13.0
Cabin,299999.0,249000.0,0.935849,0.965251,277.777778,19.0,26.0
CoOwnership,467000.0,844000.0,1.005562,0.973816,652.369138,21.0,35.5
Condominium,599950.0,609990.0,0.995935,1.0,539.215686,14.0,25.0
DeededParking,619000.0,765000.0,0.96,0.96,524.691358,12.0,31.0
Duplex,945000.0,930000.0,0.994947,1.0,562.254099,12.0,28.0
Farm,1089999.5,<NA>,<NA>,<NA>,<NA>,20.0,30.0
Loft,807000.0,316000.0,1.053333,1.053333,416.886544,10.0,35.0
ManufacturedHome,310000.0,310000.0,0.984649,0.993124,212.224782,22.5,25.0


In [132]:
sold.groupby(["PropertySubType"])[cols].median()

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
PropertySubType,,,,,,,
BoatSlip,199000.0,184500.0,0.939697,0.948293,1025.000000,29.5,17.0
Cabin,269900.0,240000.0,0.913333,0.964286,287.581699,56.0,30.0
CoOwnership,425000.0,400000.0,0.963335,0.974726,411.651469,31.0,41.0
Condominium,640000.0,628168.0,0.986207,1.000000,565.270936,29.0,28.0
DeededParking,734500.0,577500.0,0.988885,1.008885,348.407995,12.0,27.5
Duplex,939500.0,915000.0,0.983319,0.997391,545.171173,31.0,32.0
Farm,1500000.0,1425000.0,1.000000,1.000000,771.604938,12.0,29.0
Loft,697499.5,687500.0,0.982307,1.000000,515.078915,36.5,33.5
ManufacturedHome,320000.0,300000.0,0.950000,0.994975,185.303514,58.0,30.0


### CountyOrParish

We can expect market metrics to vary highly by county, as different areas have different costs of living, median income, etc. Observing which counties seem to sell properties at higher/lower prices and rates can give us a lot of insight into the micro-economics of that area. Once again, some rows is the listings segment are null because no properties from that county have been recorded as sold.

In [140]:
listings.groupby(["CountyOrParish"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
CountyOrParish,,,,,,,
Alameda,1140000.0,1225000.0,1.02345,1.024096,729.713667,10.0,23.0
Amador,532500.0,397500.0,0.925849,0.975217,218.041383,27.5,30.0
Butte,410000.0,405000.0,0.992714,1.0,272.826681,12.0,28.0
Calaveras,529450.0,487000.0,0.977914,0.983873,285.227392,12.0,30.0
Colusa,458000.0,459000.0,1.004135,1.0,229.970326,8.0,26.0
Contra Costa,875000.0,880000.0,1.0,1.000237,546.553375,9.0,22.0
Del Norte,307500.0,<NA>,<NA>,<NA>,<NA>,4.0,NaN
El Dorado,687000.0,830000.0,1.0,1.0,315.124033,14.5,29.0
Foreign Country,246500.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [142]:
sold.groupby(["CountyOrParish"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
CountyOrParish,,,,,,,
Alameda,1046500.0,1135000.0,1.019608,1.021586,700.923363,14.0,23.0
Alpine,1650000.0,1100000.0,0.666667,0.849421,267.119961,231.0,32.0
Amador,454500.0,409068.0,0.906700,0.973256,228.080552,78.0,22.5
Butte,410000.0,399000.0,0.981602,1.000000,267.348350,30.0,30.0
Calaveras,512450.0,485000.0,0.953324,0.979804,271.092635,52.0,29.0
Clark,385000.0,375000.0,0.974026,0.974026,285.605484,2.0,9.0
Colusa,454000.0,437500.0,1.000000,1.000000,215.852498,12.5,28.0
Contra Costa,819947.5,822250.0,1.000000,1.000000,521.744769,15.0,23.0
Del Norte,3250000.0,2485000.0,0.764615,1.000000,390.907661,320.0,9.0


### MLSAreaMajor

Segment analysis on MLSAreaMajor gives us similar information to that of CountyOrParish, but uses custom MLS designated districts.

In [144]:
listings.groupby(["MLSAreaMajor"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
MLSAreaMajor,,,,,,,
106 - Los Angeles,995000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
14 - South Huntington Beach,1299900.0,1299500.0,1.0,1.0,755.658095,12.0,22.0
15 - West Huntington Beach,1522508.0,1400000.0,1.0,1.0,782.647585,15.0,25.0
16 - Fountain Valley / Northeast HB,1125000.0,1165000.0,1.000002,1.0,752.444673,13.0,28.0
17 - Northwest Huntington Beach,1285000.0,1218500.0,1.0,1.0,782.040546,13.0,23.0
195 - West Wilmington,550000.0,<NA>,<NA>,<NA>,<NA>,25.0,NaN
"2 - Belmont Heights, Alamitos Heights",669000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
204 - Murrieta-Hills West,3295000.0,4000000.0,0.909091,0.909091,531.137963,100.0,35.0
206 - Murrieta-West,774950.0,705000.0,1.036765,1.036765,227.272727,29.5,13.0


In [143]:
sold.groupby(["MLSAreaMajor"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
MLSAreaMajor,,,,,,,
"1 - Belmont Shore/Park,Naples,Marina Pac,Bay Hrbr",1399950.0,1399950.0,0.984778,0.994354,808.769249,31.0,30.0
101 - North Inglewood,695000.0,673750.0,0.987440,1.000000,505.609141,39.0,35.0
102 - South Inglewood,825000.0,791000.0,0.989819,1.000000,553.145336,39.0,35.0
103 - Ladera Heights,1795000.0,1700000.0,0.961538,0.985075,648.253452,35.0,31.0
105 - Lennox,719500.0,700000.0,0.991837,0.991837,534.026465,64.0,38.0
106 - Los Angeles,1199000.0,950000.0,0.955260,0.977778,635.779817,49.0,35.0
107 - Holly Glen/Del Aire,1149000.0,1149000.0,1.000000,1.000000,751.314801,23.0,30.0
108 - North Hawthorne,819900.0,800000.0,1.000000,1.000000,560.224090,31.0,32.0
109 - Ramona/Burleigh,875000.0,850000.0,1.000000,1.000001,666.423889,20.0,30.0


### ListOfficeName

Performing analysis on ListOfficeName can help us identify which listing offices are the most successful. Note that the listings segment has many null values, because many of the listing offices in the dataset have not yet sold a property.

In [151]:
listings.groupby(["ListOfficeName"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
ListOfficeName,,,,,,,
#1 Home Sales®,412000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
#1 Realty Homes Inc.,487000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
*,825000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1 Percent Lists Desert 2 Coast,369000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1 Percent Lists Smart Rate Realty,670000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1 Percent Lists SoCal,925000.0,841730.0,0.966735,1.007918,711.166389,11.0,13.5
1 Realty,708444.0,<NA>,<NA>,<NA>,<NA>,43.0,NaN
1 STOP REALTY LLC,820000.0,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1 Team Realtors,1049500.0,712500.0,0.986364,0.986364,513.395848,11.0,29.0


In [146]:
sold.groupby(["ListOfficeName"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
ListOfficeName,,,,,,,
#1 FLAT FEE-LIBERTY REALTY,950000.0,900000.0,0.979275,0.979275,507.791510,24.0,28.0
#1 Home Sales®,649000.0,700000.0,1.000000,1.000000,347.916667,39.0,34.0
*,540000.0,565000.0,1.046296,1.046296,643.507973,21.0,31.0
1 Percent Lists SoCal,820000.0,820000.0,1.000000,1.020952,691.006601,24.0,20.0
1 Realty,943999.5,1284000.0,1.336848,0.984512,625.746926,46.0,35.5
1 Team Realtors,1150000.0,1135000.0,1.000000,0.986364,984.371972,16.0,28.5
1 Vision Real Estate,674777.0,675000.0,1.000000,1.000000,468.174645,21.0,30.0
1% LISTING FEE,639450.0,612500.0,0.976105,0.997788,420.961731,61.0,32.0
1% Listing Broker,865000.0,916000.0,1.032632,1.028902,642.447419,14.0,26.0


### BuyerOfficeName

Similarly to ListOfficeName, performing segment analysis on BuyerOfficeName helps us to identify which buying offices are the most successful. 

In [152]:
listings.groupby(["BuyerOfficeName"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
BuyerOfficeName,,,,,,,
1 Percent Lists SoCal,872500.0,841730.0,0.966735,1.007918,711.166389,20.0,13.5
1 Team Realtors,1100000.0,760000.0,0.991982,0.991982,573.874154,11.0,29.5
1 Vision Real Estate,629900.0,645000.0,1.000725,1.000725,674.816626,39.0,23.0
1% LISTING FEE,786450.0,745000.0,0.980621,0.990385,462.795815,47.0,21.5
1% Listing Broker,900000.0,1080000.0,1.166667,1.166667,732.217573,6.0,21.0
1000 Realty Corporation,744943.5,760000.0,1.025064,1.025064,427.020234,26.5,25.5
10X Realty Mortgage,1086497.5,1050000.0,1.0,1.025641,846.091861,12.0,27.0
12 Doors Real Estate,549000.0,<NA>,<NA>,<NA>,<NA>,5.0,NaN
168 Realty,1348944.0,1300777.0,1.039261,1.039261,825.608798,10.0,21.0


In [148]:
sold.groupby(["ListOfficeName"])[cols].median().head(10)

,OriginalListPrice,ClosePrice,OriginPriceRatio,PriceRatio,PricePerSquareFoot,ListToContract,ContractToClose
ListOfficeName,,,,,,,
#1 FLAT FEE-LIBERTY REALTY,950000.0,900000.0,0.979275,0.979275,507.791510,24.0,28.0
#1 Home Sales®,649000.0,700000.0,1.000000,1.000000,347.916667,39.0,34.0
*,540000.0,565000.0,1.046296,1.046296,643.507973,21.0,31.0
1 Percent Lists SoCal,820000.0,820000.0,1.000000,1.020952,691.006601,24.0,20.0
1 Realty,943999.5,1284000.0,1.336848,0.984512,625.746926,46.0,35.5
1 Team Realtors,1150000.0,1135000.0,1.000000,0.986364,984.371972,16.0,28.5
1 Vision Real Estate,674777.0,675000.0,1.000000,1.000000,468.174645,21.0,30.0
1% LISTING FEE,639450.0,612500.0,0.976105,0.997788,420.961731,61.0,32.0
1% Listing Broker,865000.0,916000.0,1.032632,1.028902,642.447419,14.0,26.0


## Summary

In [126]:
# Save the datasets with the new features

listings.to_csv("listings_week_6.csv", index=False)
sold.to_csv("sold_week_6.csv", index=False)

This week we created many key market metrics, which will help us analyze how properties sell with respect to various features. The segment analysis tables created at the end of this notebook demonstrate the beginnings of this analysis. We also further enriched our dataset by adding more information on school districts. The next step in this project will be to improve our data even further by removing outliers.